# Event Detection with `flovopy.processing.detection`

This notebook refactors the older **200 event detection** workflow so it uses
`flovopy` rather than the legacy `vsmTools` helpers.

## What changed

The original notebook used a `vsmTools` catalog object built from network
triggers. In this updated version, the workflow is **dataframe-first**:

- waveform retrieval still uses the ObsPy SDS client
- single-trace STA/LTA inspection still uses ObsPy's core trigger tools
- network event detection uses `flovopy.processing.detection`
- detected events are stored in a **pandas DataFrame**
- event windows can be exported to MiniSEED and PNG using flovopy helpers
- catalogue summaries and event-rate plots are generated directly from the dataframe

This makes the workflow easier to inspect, filter, save, and reuse in later
notebooks.


## 1. Imports and paths


In [3]:

from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
DATA_DIR = Path("~").expanduser() / "work"
SDS_DIR = DATA_DIR / "SDS"
EVENTS_DIR = DATA_DIR / "events"
CATALOG_DIR = DATA_DIR / "catalogs"

EVENTS_DIR.mkdir(parents=True, exist_ok=True)
CATALOG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SDS archive client
# ------------------------------------------------------------------
from obspy.clients.filesystem.sds import Client
sdsclient = Client(str(SDS_DIR))


## 2. Import the flovopy detection tools

This notebook assumes `flovopy` is installed in your active environment. For
development workflows, the usual pattern is something like:

```bash
pip install -e /path/to/flovopy
```

If needed, you can also append a local repository path below.


In [4]:

# Optional: uncomment and edit if flovopy is not already importable
# sys.path.append("/path/to/flovopy")

from flovopy.processing.detection import (
    detect_network_event,
    extract_triggers_to_dataframe,
    export_events_from_catalogue,
    filter_events_df,
    plot_event_rate,
    plot_stream_with_event_markers,
    real_time_optimization,
    run_coincidence_trigger_dataframe,
)


## 3. Detection parameters

These values follow the broad-event settings you were already using in the old
notebook. The `real_time_optimization()` helper gives a reasonable starting
point for broad event classes.


In [5]:

# Suggested broad-band real-time parameters
sta_secs, lta_secs, thresh_on, thresh_off_ratio, freqmin, freqmax, corners = real_time_optimization("all")

# In the original notebook, threshOFF was used as an absolute value (1.2),
# not the normalized ratio returned by real_time_optimization().
thresh_off = 1.2

thresh_stations = 3
max_secs = 120.0
pretrig = 10.0
posttrig = 20.0

print(f"STA = {sta_secs:.2f} s")
print(f"LTA = {lta_secs:.2f} s")
print(f"Trigger ON = {thresh_on:.2f}")
print(f"Trigger OFF = {thresh_off:.2f}")
print(f"Bandpass = {freqmin:.1f}–{freqmax:.1f} Hz")
print(f"Minimum coincident channels = {thresh_stations}")


STA = 2.30 s
LTA = 11.50 s
Trigger ON = 2.40
Trigger OFF = 1.20
Bandpass = 1.5–12.0 Hz
Minimum coincident channels = 3


## 4. Small helper functions

These helpers keep the later notebook cells compact and readable.


In [6]:

def get_stream(stime, etime, network="MV", station="*", location="*", channel="?HZ"):
    """Read a time window from the SDS archive."""
    st = sdsclient.get_waveforms(network, station, location, channel, stime, etime)
    return st

def drop_short_traces(st, min_seconds):
    """Remove traces that are too short to support the chosen trigger length."""
    st_out = st.copy()
    for tr in list(st_out):
        min_npts = min_seconds * tr.stats.sampling_rate
        if tr.stats.npts < min_npts:
            st_out.remove(tr)
    return st_out

def preprocess_detection_stream(st, freqmin, freqmax, corners=3):
    """Basic preprocessing for network event detection."""
    st2 = st.copy()
    if len(st2):
        st2.detrend("linear")
        st2.taper(max_percentage=0.01, type="cosine")
        st2.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=corners)
    return st2

def detect_events_for_window(
    st,
    threshold_on=thresh_on,
    threshold_off=thresh_off,
    sta_seconds=sta_secs,
    lta_seconds=lta_secs,
    min_channels=thresh_stations,
    pretrigger_seconds=pretrig,
    posttrigger_seconds=posttrig,
    outdir=EVENTS_DIR,
    write_mseed=False,
):
    """Run coincidence triggering on a preprocessed stream and return a dataframe."""
    if len(st) == 0:
        return pd.DataFrame()

    df = run_coincidence_trigger_dataframe(
        st,
        trigger_type="classicstalta",
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        threshold_on=threshold_on,
        threshold_off=threshold_off,
        min_channels=min_channels,
        pretrigger_seconds=pretrigger_seconds,
        posttrigger_seconds=posttrigger_seconds,
        write_mseed=write_mseed,
        outdir=str(outdir),
        make_plots=False,
        filename_safe=True,
        details=True,
        max_trigger_length=max_secs,
    )
    return df

def summarize_detection_df(df):
    """Print a concise summary of an event dataframe."""
    if df is None or len(df) == 0:
        print("No events detected.")
        return
    print(f"Detected {len(df)} events")
    cols = [c for c in ['on_time', 'off_time', 'duration_s', 'coincidence_sum', 'snr_rms', 'n_seed_ids'] if c in df.columns]
    display(df[cols].head())


## 5. Inspect a few hours of data

A dayplot is still a good way to pick sensible trigger parameters before doing
larger batch detection.


In [9]:

st = get_stream(
    obspy.UTCDateTime(2003, 7, 11, 0, 0, 0),
    obspy.UTCDateTime(2003, 7, 11, 6, 0, 0),
    station="MBLG",
    location="--",
    channel="SHZ",
)

st.plot(type="dayplot", interval=10, vertical_scaling_range=3e3);


IndexError: Empty stream object

The strong repetition here is characteristic of **drumbeat seismicity**.


## 6. Optional: write the trace to audio

This cell is optional and platform-dependent. It is left here because it was
part of the original notebook and can still be useful for quick qualitative
inspection.


In [ ]:

wav_audio_file = DATA_DIR / "MV20030711.wav"
st[0].write(str(wav_audio_file), rescale=True, format="WAV", framerate=6000 * 2)

print(f"Wrote {wav_audio_file}")
# On macOS you might use:
# os.system(f'open "{wav_audio_file}"')
# On Linux you might use:
# os.system(f'xdg-open "{wav_audio_file}"')


## 7. Example 1 — inspect STA/LTA behaviour on one hour of data

We first look at one hour of data and inspect the STA/LTA characteristic
function on a single trace.


In [ ]:

from obspy.signal.trigger import classic_sta_lta, plot_trigger

stime = obspy.UTCDateTime("2003193T")
etime = stime + 3600

st = get_stream(stime, etime)
st = drop_short_traces(st, max_secs)

print(f"Number of traces in raw stream: {len(st)}")
st.plot(equal_scale=False);


In [ ]:

st_single = preprocess_detection_stream(st, freqmin=freqmin, freqmax=freqmax, corners=corners)

tr_index = -1
tr = st_single[tr_index]

Fs = int(round(tr.stats.sampling_rate))
sta_samples = int(sta_secs * Fs)
lta_samples = int(lta_secs * Fs)

print(f"Trace selected: {tr.id}")
print(f"Sampling rate: {Fs} Hz")

cft = classic_sta_lta(tr.data, sta_samples, lta_samples)
plot_trigger(tr, cft, thresh_on, thresh_off)
plt.show()


The upper panel shows the waveform with trigger-on and trigger-off markers,
while the lower panel shows the STA/LTA ratio. This is still the quickest way
to check whether the chosen parameters are sensible on a representative trace.


## 8. Example 2 — detect coincident network events in one hour

Now we use the flovopy helpers to detect network events and immediately convert
the results into a dataframe.


In [ ]:

st_det = preprocess_detection_stream(st, freqmin=freqmin, freqmax=freqmax, corners=corners)

trig, ontimes, offtimes = detect_network_event(
    st_det,
    minchans=thresh_stations,
    threshon=thresh_on,
    threshoff=thresh_off,
    sta=sta_secs,
    lta=lta_secs,
    algorithm="classicstalta",
    join_within=5.0,
    min_duration=0.5,
    verbose=False,
)

if trig is None:
    print("No triggers found.")
else:
    print(f"Number of network triggers: {len(trig)}")
    print("First trigger dictionary:")
    from pprint import pprint
    pprint(trig[0])


In [ ]:

if trig:
    event_df = extract_triggers_to_dataframe(
        st_det,
        trig,
        pretrigger_seconds=pretrig,
        posttrigger_seconds=posttrig,
        write_mseed=False,
        outdir=str(EVENTS_DIR),
        make_plots=False,
        filename_safe=True,
    )
else:
    event_df = pd.DataFrame()

summarize_detection_df(event_df)


In [ ]:

if len(event_df):
    _fig, _axes = plot_stream_with_event_markers(
        st_det,
        event_df,
        on_col="on_time",
        off_col="off_time",
        show=True,
        shade_events=True,
        equal_scale=False,
    )


### Filter the hourly detections

A dataframe-first workflow makes it easy to remove obvious weak or noisy
detections before exporting or aggregating them.


In [ ]:

event_df_filt = filter_events_df(
    event_df,
    min_snr=1.2,
    min_duration_s=0.5,
    min_seed_ids=thresh_stations,
)

print(f"Raw detections:      {len(event_df)}")
print(f"Filtered detections: {len(event_df_filt)}")
display(event_df_filt.head())


### Optional: export the filtered events

This can write MiniSEED windows and quick-look PNGs for each detected event.


In [ ]:

export_df = export_events_from_catalogue(
    event_df_filt,
    base_outdir=str(EVENTS_DIR / "example_hour"),
    write_mseed=False,
    write_png=False,
    st_continuous=st_det,
)

display(export_df.head())


### Plot event rate for the one-hour example


In [ ]:

plot_event_rate(
    event_df_filt,
    bin_size="1min",
    title="Detected event rate for one example hour",
    show=True,
);


## 9. Example 3 — run coincidence triggering over multiple days

The original notebook looped over each hour, created a catalog object, wrote
event files, and concatenated catalogs. Here we keep the same basic loop, but
we collect per-hour dataframes and concatenate them at the end.


In [ ]:

# Detection settings for the multi-day run
jday_start = 190
jday_stop = 197   # stop is exclusive, so this covers days 190-196
hourly_catalogues = []

for jday in range(jday_start, jday_stop):
    print(f"Julian day {jday}: ", end="")
    for hour in range(24):
        stime = obspy.UTCDateTime(f"2003{jday}T") + hour * 3600
        print(f"{hour:02d}", end=" " if hour < 23 else "\n")

        st = get_stream(stime, stime + 3600)
        if len(st) == 0:
            continue

        st = drop_short_traces(st, max_secs)
        if len(st) == 0:
            continue

        st_det = preprocess_detection_stream(st, freqmin=freqmin, freqmax=freqmax, corners=corners)

        df_hour = detect_events_for_window(
            st_det,
            threshold_on=thresh_on,
            threshold_off=thresh_off,
            sta_seconds=sta_secs,
            lta_seconds=lta_secs,
            min_channels=thresh_stations,
            pretrigger_seconds=pretrig,
            posttrigger_seconds=posttrig,
            outdir=EVENTS_DIR,
            write_mseed=False,
        )

        if len(df_hour):
            df_hour = df_hour.copy()
            df_hour["window_start"] = pd.Timestamp(stime.datetime)
            df_hour["jday"] = jday
            df_hour["hour"] = hour
            hourly_catalogues.append(df_hour)

if hourly_catalogues:
    catalog_df = pd.concat(hourly_catalogues, ignore_index=True)
else:
    catalog_df = pd.DataFrame()

print(f"Total detected events: {len(catalog_df)}")


## 10. Inspect the combined dataframe


In [ ]:

if len(catalog_df):
    display(catalog_df.head())
    print(catalog_df.dtypes)
else:
    print("catalog_df is empty")


The old notebook used a custom catalog object. In this refactored version,
`catalog_df` is now the main catalogue product. It is easier to filter, save,
reload, and pass into later notebooks.


## 11. Save the combined catalogue

Because the current workflow is dataframe-based, we save the detections as CSV
and Pickle rather than as the older custom event catalog / QuakeML object.


In [ ]:

csv_path = CATALOG_DIR / "catalog_MV_20030712_flovopy.csv"
pkl_path = CATALOG_DIR / "catalog_MV_20030712_flovopy.pkl"

catalog_df.to_csv(csv_path, index=False)
catalog_df.to_pickle(pkl_path)

print(csv_path)
print(pkl_path)
print(list(CATALOG_DIR.iterdir()))


## 12. Plot the event catalogue

These plots replace the old `catalogObj.plot_eventrate(...)` workflow.


In [ ]:

plot_event_rate(
    catalog_df,
    bin_size="1H",
    title="Detected event rate (1-hour bins)",
    show=True,
);


In [ ]:

plot_event_rate(
    catalog_df,
    bin_size="10min",
    title="Detected event rate (10-minute bins)",
    show=True,
);


In [ ]:

catalog_df.plot.scatter(
    x="on_time",
    y="duration_s",
    s=2,
    rot=90,
    xlabel="Date",
    ylabel="Trigger duration (s)",
    title="Trigger duration through time",
);
plt.show()


In [ ]:

ax = catalog_df.plot.scatter(
    x="on_time",
    y="duration_s",
    s=2,
    rot=90,
    xlabel="Date",
    ylabel="Trigger duration (s)",
    title="Trigger duration through time (zoomed)",
)
ax.set_ylim([0, 20])
plt.show()


In [ ]:

if "coincidence_sum" in catalog_df.columns:
    ax = catalog_df.plot.scatter(
        x="on_time",
        y="coincidence_sum",
        s=2,
        rot=90,
        xlabel="Date",
        ylabel="Coincidence sum",
        title="Coincidence sum through time",
    )
    plt.show()


In the legacy notebook, the custom event catalog also exposed a crude magnitude.
The dataframe workflow provided by `flovopy.processing.detection` does **not**
currently build that same catalog-object magnitude field, so this notebook
focuses on directly available measures such as:

- trigger duration
- RMS-based SNR
- number of contributing traces
- coincidence sum


## 13. Inspect a period of elevated activity

This reproduces the original qualitative inspection step on a time window where
activity increases strongly.


In [ ]:

st2 = get_stream(
    obspy.UTCDateTime(2003, 7, 12, 8, 0, 0),
    obspy.UTCDateTime(2003, 7, 13, 2, 0, 0),
    station="MBWH",
    location="",
    channel="SHZ",
)
st2.plot(type="dayplot", interval=10, vertical_scaling_range=8e3);


## 14. Zoom in on the event-rate plot

Because we now store detections in a dataframe, time filtering is done with
ordinary pandas logic.


In [ ]:

time_min = pd.Timestamp(2003, 7, 12, 0, 0, 0)
time_max = pd.Timestamp(2003, 7, 14, 12, 0, 0)

catalog_zoom = catalog_df[
    (pd.to_datetime(catalog_df["on_time"]) >= time_min)
    & (pd.to_datetime(catalog_df["on_time"]) <= time_max)
].copy()

plot_event_rate(
    catalog_zoom,
    bin_size="10min",
    title="Detected event rate (zoomed window)",
    show=True,
);


## 15. Suggested next steps

This notebook now gives you a clean `flovopy`-based event-detection workflow.
A natural follow-on notebook could add:

1. event classification
2. phase picking
3. improved magnitude or energy measures
4. event family similarity / template matching
5. conversion of the dataframe catalogue into a richer ObsPy `Catalog` if you decide you want that later
